# Week 4 — Predictive Modeling & Performance Evaluation

**Financial Data Analysis Internship | AAPL**

This notebook continues the Week 1–3 AAPL project using `data/processed/AAPL_cleaned.csv`. It builds next-day Close and next-day return targets, uses a chronological 80/20 split, compares a persistence baseline with linear regression models, calculates R²/MSE/RMSE/MAE, and writes the Week 4 figures and prediction CSVs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'AAPL_cleaned.csv'
FIG = ROOT / 'figures' / 'week4'
REP = ROOT / 'reports'
FIG.mkdir(parents=True, exist_ok=True); REP.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA, parse_dates=['Date']).set_index('Date').sort_index()
df['SMA_5'] = df['Close'].rolling(5).mean()
df['SMA_10'] = df['Close'].rolling(10).mean()
df['Target_Close_Next'] = df['Close'].shift(-1)
df['Target_Return_Next'] = df['Daily_Return'].shift(-1)
FEATURES = ['Close','Volume','Rolling_Volatility_20D','Intraday_Range_Pct','Daily_Return','SMA_5','SMA_10']
RETURN_FEATURES = ['Volume','Rolling_Volatility_20D','Intraday_Range_Pct','Daily_Return','Volume_Change_Pct']
model = df.dropna(subset=FEATURES + ['Target_Close_Next','Target_Return_Next'])
split = int(len(model) * 0.8)
train, test = model.iloc[:split], model.iloc[split:]
print(f'Train: {len(train)} rows; Test: {len(test)} rows')
print(train.index.min().date(), 'to', train.index.max().date())
print(test.index.min().date(), 'to', test.index.max().date())

## Model evaluation

The primary target is next-day Close. The naive persistence forecast uses today's Close as tomorrow's prediction. Model 1 uses Close only; Model 2 uses the seven-feature set. A secondary Model 3 predicts next-day Daily_Return using five price/volume-derived features.

In [ ]:
def metrics(y, p):
    mse = mean_squared_error(y, p)
    return {'R2': r2_score(y,p), 'MSE': mse, 'RMSE': np.sqrt(mse), 'MAE': mean_absolute_error(y,p)}

ytr, yte = train['Target_Close_Next'].to_numpy(), test['Target_Close_Next'].to_numpy()
naive = metrics(yte, test['Close'].to_numpy())
m1 = LinearRegression().fit(train[['Close']], ytr)
m1_test = metrics(yte, m1.predict(test[['Close']]))
m2 = LinearRegression().fit(train[FEATURES], ytr)
p2 = m2.predict(test[FEATURES])
m2_test = metrics(yte, p2)

rtr, rte = train['Target_Return_Next'].to_numpy(), test['Target_Return_Next'].to_numpy()
m3 = LinearRegression().fit(train[RETURN_FEATURES], rtr)
p3 = m3.predict(test[RETURN_FEATURES])
m3_test = metrics(rte, p3)

results = pd.DataFrame([naive, m1_test, m2_test], index=['Naive persistence','Model 1 — Close only','Model 2 — Multivariate'])
display(results)
print('Model 3 return test metrics:', m3_test)

In [ ]:
# Save the two prediction datasets used by the Week 4 report.
pred2 = test[['Close','Target_Close_Next']].copy()
pred2['Predicted_Close_Next'] = p2
pred2['Residual'] = pred2['Target_Close_Next'] - pred2['Predicted_Close_Next']
pred2.to_csv(REP / 'model2_test_predictions.csv')
pred3 = test[['Target_Return_Next']].copy()
pred3['Predicted_Return_Next'] = p3
pred3.to_csv(REP / 'model3_test_predictions.csv')
print('Prediction CSVs saved.')

## Reproducibility

The same pipeline is also provided in `src/week4_predictive_modeling.py`. The modeling window is chronological, with no random shuffling, and targets are created with `shift(-1)` so only information available by the end of day *t* is used to predict day *t+1*.